In [1]:
from utils.orthanc_utils import *
from utils.db_utils import *

from pydicom.uid import generate_uid, ExplicitVRLittleEndian

def get_volumes(dicoms):    
    df = pd.DataFrame(
        [{"SliceLocation": d.SliceLocation, "InstanceNumber": d.InstanceNumber, "PixelArray": d.pixel_array} for d in dicoms]
    ).sort_values(["SliceLocation", "InstanceNumber"])

    d0 = dicoms[0]
    pixel_spacing = float(d0.PixelSpacing[0])
    thickness = float(getattr(d0, "SpacingBetweenSlices", getattr(d0, "SliceThickness", np.nan)))
    voxel_size = (pixel_spacing ** 2 * thickness) / 1000
    
    array_4d = []
    for _, slice_df in df.groupby("SliceLocation"):
        time_array = []
        for _, time_df in slice_df.groupby("InstanceNumber"):
            pixel_array = time_df["PixelArray"].iloc[0]
            time_array.append(pixel_array)

        slice_array = np.stack(time_array, axis=-1)
        array_4d.append(slice_array)

    array_4d = np.stack(array_4d, axis=-2)

    labels = {500: "lv", 1000: "rv", 1500: "lv_myo", 2000: "rv_myo"}
    volumes = {
        name: np.sum(array_4d == val, axis=(0, 1, 2)) * voxel_size
        for val, name in labels.items()
    }

    return volumes["lv"], volumes["rv"], volumes["lv_myo"], volumes["rv_myo"]

def send_series_to_orthanc(new_array, old_dcm):
    new_series_uid = generate_uid()
    for i in range(len(new_array)):
        old_dcm[i].SeriesInstanceUID = new_series_uid
        old_dcm[i].SeriesDescription = 'Roundel Processed Series'
        old_dcm[i].PixelData = new_array[i].astype(np.uint16).tobytes()
        old_dcm[i].SOPInstanceUID = generate_uid()
        old_dcm[i].file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
        old_dcm[i].is_little_endian = True
        old_dcm[i].is_implicit_VR = False
        buffer = io.BytesIO()
        old_dcm[i].save_as(buffer)
        buffer.seek(0)
        upload = SESSION.post(f"{ORTHANC}/instances",data=buffer.read(),auth=AUTH,headers={"Content-Type": "application/dicom", "Expect": ""})
        upload.raise_for_status()
    return new_series_uid


db = TinyDB('image_clasp_db.json')
StudyQuery = Query()
SeriesQuery = Query()
for study in db.search(StudyQuery.series.any(SeriesQuery.series_description == 'DL Processed')):
    dl_processed_series_list = [series for series in study.get("series", []) if series.get("series_description") == "DL Processed"]

    image_dicoms = [d for s in dl_processed_series_list for d in fetch_dicoms_for_series(s['associated_orthanc_id'])]
    mask_dicoms = [d for s in dl_processed_series_list for d in fetch_dicoms_for_series(s['orthanc_series_id'])]
    lv_volume, rv_volume, lv_myo_volume, rv_myo_mask = get_volumes(mask_dicoms)

    masked_images = [image.pixel_array * (mask.pixel_array >0) for image, mask in zip(image_dicoms, mask_dicoms) ]
    new_orthanc_id = send_series_to_orthanc(masked_images, image_dicoms)
